In [3]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Load the UCI dataset
df = pd.read_csv("../data/PhishingData.csv")

# Drop the 'index' or 'id' column if it exists
if "index" in df.columns:
    df = df.drop(columns=["index"])

# The target column in this dataset is typically the last column ('Result')
target_col = df.columns[-1]
print("Target column:", target_col)

X = df.drop(columns=[target_col])
# UCI dataset labels are usually 1 (legit) and -1 (phishing), let's convert to 1 (phishing) and 0 (legit)
y = (df[target_col] == -1).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

lr = make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)

for name, m in [("LogReg", lr), ("RandomForest", rf)]:
    print(f"\nTraining {name}...")
    m.fit(X_tr, y_tr)
    print(f"=== {name} Report ===")
    print(classification_report(y_te, m.predict(X_te), target_names=["legit","phishing"]))

# Save the Random Forest model and column names
os.makedirs("../models", exist_ok=True)
joblib.dump(rf, "../models/url_model.pkl")
joblib.dump(list(X.columns), "../models/url_feature_names.pkl")
print("\nSuccessfully saved url_model.pkl and url_feature_names.pkl!")

Target column: Result

Training LogReg...
=== LogReg Report ===
              precision    recall  f1-score   support

       legit       0.93      0.94      0.94      1231
    phishing       0.93      0.92      0.92       980

    accuracy                           0.93      2211
   macro avg       0.93      0.93      0.93      2211
weighted avg       0.93      0.93      0.93      2211


Training RandomForest...
=== RandomForest Report ===
              precision    recall  f1-score   support

       legit       0.97      0.98      0.97      1231
    phishing       0.97      0.97      0.97       980

    accuracy                           0.97      2211
   macro avg       0.97      0.97      0.97      2211
weighted avg       0.97      0.97      0.97      2211


Successfully saved url_model.pkl and url_feature_names.pkl!
